# Week 51, Day 1: Meet the data: 18 mice in the IBL task

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast = d["contrast"].astype(float)      # signed: negative left, positive right
choice = d["choice"].astype(float)          # 1 right, 0 left, -1 no choice
reward = d["reward"].astype(int)
p_left = d["p_left"].astype(float).round(2) # 0.5, 0.2 (right block) or 0.8 (left block)
rt = d["rt"].astype(float)                  # seconds from stimulus onset to response
included = d["included"]; mice, labs = d["mice"], d["labs"]
ok = (choice >= 0) & included & np.isfinite(rt)   # trials with a choice and a response time, kept by the experimenters
print(f"{len(choice)} trials, {len(mice)} mice from {len(set(labs))} laboratories, {ok.mean():.1%} usable")
print("contrasts:", np.unique(contrast))

### Exercise: A first look at 70,000 trials

Count the usable trials of each mouse in `n_trials`. Then, for each absolute contrast in `levels`, compute the fraction of usable trials that were rewarded (`acc`) and their median response time (`med_rt`).

In [ ]:
levels = np.array([0, 0.0625, 0.125, 0.25, 1])
n_trials = ...
acc = ...
med_rt = ...
print("usable trials per mouse:", n_trials.min(), "to", n_trials.max())
for a, p, t in zip(levels, acc, med_rt):
    print(f"contrast {a:6.4f}: {p:.1%} correct, median response time {1000 * t:.0f} ms")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(levels, acc, "o-"); ax[0].set_xscale("symlog", linthresh=0.05); ax[0].set_xlabel("contrast"); ax[0].set_ylabel("fraction correct")
ax[1].plot(levels, 1000 * med_rt, "o-"); ax[1].set_xscale("symlog", linthresh=0.05); ax[1].set_xlabel("contrast"); ax[1].set_ylabel("median RT (ms)"); plt.show()

In [ ]:
import numpy as np
def test_counts():
    "n_trials counts the usable trials of each mouse"
    assert n_trials.shape == (18,) and np.all(n_trials == np.bincount(mouse[ok], minlength=18))
def test_acc():
    "acc holds the fraction rewarded at each absolute contrast, over usable trials"
    exp = np.array([reward[ok & (np.abs(contrast) == a)].mean() for a in levels])
    assert np.allclose(acc, exp)
def test_rt():
    "med_rt holds the median response time at each absolute contrast"
    exp = np.array([np.median(rt[ok & (np.abs(contrast) == a)]) for a in levels])
    assert np.allclose(med_rt, exp)

test_counts()
test_acc()
test_rt()
print("All checks passed")

### Exercise: How far does the block move each mouse?

Write psychometric(params, c) for params = (mu, sigma, gamma, lam). The negative log-likelihood and the fitting loop are written for you: fill in mu for the left block minus mu for the right block, for each mouse, in shift.

In [ ]:
from scipy.special import erf
from scipy.optimize import minimize

def psychometric(params, c):
    mu, sigma, gamma, lam = params
    return ...

def nll(params, c, y):
    p = np.clip(psychometric(params, c), 1e-9, 1 - 1e-9)
    return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def fit_psy(c, y):
    res = minimize(nll, [0, 0.1, 0.05, 0.05], args=(c, y), method="L-BFGS-B",
                   bounds=[(-0.5, 0.5), (0.005, 1), (0, 0.5), (0, 0.5)])
    return res.x

fits = {}
shift = np.zeros(18)
for m in range(18):
    for p in [0.2, 0.8]:
        k = ok & (mouse == m) & (p_left == p)
        fits[m, p] = fit_psy(contrast[k], choice[k])
    shift[m] = ...
print("bias shift (left block minus right block) per mouse:", np.round(shift, 3))
print(f"median shift {np.median(shift):.3f}; positive for {np.sum(shift > 0)} of 18 mice")

In [ ]:
import numpy as np
def test_psy():
    "psychometric matches the formula with an erf"
    from scipy.special import erf
    c = np.linspace(-1, 1, 9); p = [0.05, 0.2, 0.1, 0.05]
    assert np.allclose(psychometric(p, c), 0.1 + 0.85 * 0.5 * (1 + erf((c - 0.05) / (0.2 * np.sqrt(2)))))
def test_shift():
    "shift has one value per mouse and every mouse is shifted towards the likely side"
    assert shift.shape == (18,) and np.all(shift > 0)
def test_size():
    "the median shift lies between 0.03 and 0.15"
    assert 0.03 < np.median(shift) < 0.15

test_psy()
test_shift()
test_size()
print("All checks passed")

### Exercise: Are choices that agree with the block faster?

Mark each choice as congruent if it goes towards the block's likely side (right in blocks with p_left 0.2, left in blocks with p_left 0.8). Then, for zero-contrast trials in biased blocks, store each mouse's median RT for congruent and incongruent choices in med and count the mice that are faster when congruent.

In [ ]:
biased = p_left != 0.5
congruent = ...
k = ok & biased & (contrast == 0)
med = np.zeros((18, 2))
for m in range(18):
    med[m] = ...
n_faster = ...
print(f"pooled medians: congruent {1000 * np.median(rt[k & congruent]):.0f} ms, incongruent {1000 * np.median(rt[k & ~congruent]):.0f} ms")
print(f"{n_faster} of 18 mice are faster when their choice agrees with the block")
print(f"responses slower than 2 s: {np.mean(rt[ok] > 2):.1%} of trials")

In [ ]:
import numpy as np
def test_cong():
    "congruent marks choices towards the likely side, in biased blocks only"
    exp = (choice == 1) == (p_left == 0.2)
    assert np.all(congruent[biased] == exp[biased])
def test_mice():
    "med holds each mouse's median zero-contrast RT for congruent and incongruent choices"
    k = ok & biased & (contrast == 0)
    m3 = np.median(rt[k & (mouse == 3) & ~congruent])
    assert med.shape == (18, 2) and abs(med[3, 1] - m3) < 1e-9
def test_count():
    "n_faster counts mice whose congruent median is shorter"
    assert n_faster == np.sum(med[:, 0] < med[:, 1]) and n_faster >= 14

test_cong()
test_mice()
test_count()
print("All checks passed")